# Explainable Machine Learning for Personalized Next-Topic Recommendation Based on Student Learning Performance

**Starter notebook for the project report**

**Important:** This notebook uses a small *illustrative synthetic dataset* so that the workflow can be tested. These are not real student records and the outputs must not be reported as experimental findings. For the final project, replace the sample data with the dataset you actually select and document its source.

## Workflow
1. Load and inspect student performance data
2. Explore scores and topic completion
3. Train a simple Decision Tree to predict a learning-support category
4. Evaluate the model
5. Inspect feature importance as a basic explanation

This is a starter demonstration, not a validated next-topic recommender. A true next-topic recommendation experiment needs real data with topic history and a defensible target for the next topic.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay
from sklearn.inspection import permutation_importance

print("Libraries loaded successfully.")

## 1. Create illustrative sample data

The sample data below is generated only to demonstrate the code. It is not taken from a real dataset.

In [ ]:
rng = np.random.default_rng(42)
n = 120

df = pd.DataFrame({
    "quiz_score": rng.integers(20, 101, n),
    "assessment_score": rng.integers(20, 101, n),
    "topic_completion_pct": rng.integers(20, 101, n),
    "previous_attempts": rng.integers(1, 6, n)
})

# Demonstration target only: labels are generated from simple rules.
df["support_category"] = np.where(
    (df["quiz_score"] < 50) | (df["assessment_score"] < 50),
    "Revision",
    np.where(
        (df["topic_completion_pct"] < 60) | (df["previous_attempts"] >= 4),
        "Practice",
        "Next Topic"
    )
)

df.head()

## 2. Inspect the data

In [ ]:
print("Dataset shape:", df.shape)
print("\nData types and non-null counts:")
df.info()
print("\nMissing values:")
print(df.isnull().sum())
print("\nTarget category counts:")
print(df["support_category"].value_counts())

## 3. Visualize student performance

In [ ]:
plt.figure(figsize=(7, 4))
plt.hist(df["quiz_score"], bins=10, edgecolor="black")
plt.title("Illustrative Distribution of Quiz Scores")
plt.xlabel("Quiz score")
plt.ylabel("Number of sample records")
plt.tight_layout()
plt.show()

df.groupby("support_category")[["quiz_score", "assessment_score"]].mean().plot(
    kind="bar", figsize=(8, 4)
)
plt.title("Average Scores by Demonstration Support Category")
plt.ylabel("Average score")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 4. Train a basic Decision Tree

The target here is a demonstration support category, **not the identity of the next topic**.

In [ ]:
features = [
    "quiz_score",
    "assessment_score",
    "topic_completion_pct",
    "previous_attempts"
]
X = df[features]
y = df["support_category"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = DecisionTreeClassifier(max_depth=4, random_state=42)
model.fit(X_train, y_train)
predictions = model.predict(X_test)

print("Accuracy on illustrative synthetic test split:", round(accuracy_score(y_test, predictions), 3))
print("\nClassification report:")
print(classification_report(y_test, predictions, zero_division=0))

## 5. Confusion matrix

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test, predictions, xticks_rotation="vertical"
)
plt.title("Confusion Matrix — Illustrative Synthetic Data")
plt.tight_layout()
plt.show()

## 6. Explain the model with feature importance

In [ ]:
importance = pd.Series(
    model.feature_importances_, index=features
).sort_values()

importance.plot(kind="barh", figsize=(7, 4))
plt.title("Decision Tree Feature Importance — Illustrative Data")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

print("Feature importance values:")
print(importance.sort_values(ascending=False))

## 7. Try one example recommendation

This is a demonstration of the support-category classifier. It does not select a named curriculum topic.

In [ ]:
new_student = pd.DataFrame([{
    "quiz_score": 42,
    "assessment_score": 55,
    "topic_completion_pct": 70,
    "previous_attempts": 2
}])

predicted_category = model.predict(new_student[features])[0]
print("Demonstration support category:", predicted_category)

## 8. Replace the sample data for the final project

For the final experiment:
1. Select a suitable real dataset and record its official source URL.
2. Replace the synthetic data section with code that loads the chosen dataset.
3. Map the real columns to features and define a valid next-topic target.
4. Compare the model with a fixed-sequence baseline.
5. Evaluate recommendation quality with suitable metrics such as Precision@K and Recall@K.
6. Report only the values produced by the real experiment.
7. If using SHAP, install/import it and run it on the final trained model.

### Candidate dataset source (not yet confirmed as used)
UCI Student Performance: https://archive.ics.uci.edu/dataset/320/student+performance

**Important limitation:** UCI Student Performance contains student demographic and academic performance data; it does not directly provide a learning-topic sequence for next-topic recommendation. Do not describe it as the final recommendation dataset unless you have a suitable way to define and justify the recommendation target.